# Conversion for Mobile Deployment


- PyTorch → ONNX (in Prototype 10)
- ONNX → TensorFlow → TFLite + INT8 Quantization

Requirements
```
convlstm.onnx
convlstm.onnx.data
Compressed dataset
```

References
- https://github.com/DanieliusKr/onnx-example/blob/main/onnx_example.ipynb
- https://www.geeksforgeeks.org/machine-learning/convert-pytorch-model-to-tf-lite-with-onnx-tf/

### Import

In [1]:
# Check Python version
!python --version

Python 3.12.13


In [2]:
try:
    import google.colab
    IN_COLAB = True

    !pip install onnx2tf
    """
    !pip install onnx
    !pip install onnx_tf

    !pip install tensorflow-addons
    !pip install tensorflow-probability
    """
    from google.colab import files
except ImportError:
    IN_COLAB = False

In [3]:
import torch as torch
import torch.nn as nn
import os
import random
import numpy as np
import cv2
import pandas as pd
import scipy.stats as ss

import onnx2tf
import tensorflow as tf
import torch.onnx
import onnx
import sys
# from onnx_tf.backend import prepare

## Conversion to TensorFlow and TFLite

Initialize:
```
dataset = "_"
```

In [4]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
HEIGHT = 128
WIDTH = 128

def data_gen():
    # Initialize dataset folder
    dataset = "Dataset_750"
    if IN_COLAB:
      !unzip -q /content/{dataset} -d /content/

      dataset = os.path.join("/content", dataset)
      if os.path.exists(dataset):
        print("Dataset unzipped.")

    return get_input(DEVICE, False, dataset)

def get_input(DEVICE, isRandom, data) -> torch.Tensor:
    """ Generate random input data with the same shape as the model's expected input """
    if isRandom:
        # Generate random video
        # Timesteps, Channels, Height, Width
        video_tensor = torch.rand(20, 3, 128, 128).to(DEVICE)
        # Generate random intent
        intent_direction = random.randint(-1, 2)
        # Generate random intent position
        intent_position = random.randint(0, 9)

        # Convert video to frames
        frames = []
        for i in range(video_tensor.shape[0]):
            frame_tensor = video_tensor[i]
            frame_tensor= get_intent_torch(intent_position, i, intent_direction, frame_tensor, 1)
            frames.append(frame_tensor)

        video_tensor = torch.stack(frames, dim=0)
        # Adds batch dimension to the tensor -> Batch, Timesteps, Channels, Height, Width
        video_tensor = video_tensor.unsqueeze(0)

        return video_tensor
    else:
        if IN_COLAB:
          npy_file = "/content/tflite_int8_data.npy"
        else:
          npy_file = "tflite_int8_data.npy"
        representative_data = []

        """
        data_train = os.path.join(data, "train")
        data_train_vid = os.path.join(data_train, "videos")
        data_train_lbl = os.path.join(data_train, "labels")
        labels = [f for f in os.listdir(data_train_lbl) if f.endswith('.csv')]
        """
        data_train_vid = os.path.join(data, "videos")
        data_train_lbl = os.path.join(data, "labels")
        labels = [f for f in os.listdir(data_train_lbl) if f.endswith('.csv')]

        print(f"Number of files in {data_train_vid}: {len(os.listdir(data_train_vid))}")
        for i, video_name in enumerate(os.listdir(data_train_vid)):
        # Representative dataset for Integer Quantization
            if i < 100:
                intent_position = get_intent_position()
                df = pd.read_csv(os.path.join(data_train_lbl, labels[i]))
                intent = get_intent(intent_position, df)

                video_path = os.path.join(data_train_vid, video_name)
                cap = cv2.VideoCapture(video_path)
                frames = []

                for i in range(20):
                    ret, frame = cap.read()
                    if not ret:
                        frame_tensor = torch.zeros((3, HEIGHT, WIDTH))
                    else:
                        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                        """
                        if self.transforms:
                            frame = Image.fromarray(frame)
                            frame_tensor = self.transforms(frame)
                        else:
                        """
                        frame_tensor = torch.from_numpy(frame).permute(2, 0, 1).float() / 255.0

                    frame_tensor = get_intent_torch(intent_position, i, intent, frame_tensor, 128)
                    frames.append(frame_tensor)

                cap.release()
                video_tensor = torch.stack(frames, dim=0)

                # Global Average Pooling to fit the 1x1 dimensions
                global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))
                video_tensor = global_avg_pool(video_tensor)
                representative_data.append(video_tensor)

        print(f"Number of items in the representative_data: {len(representative_data)}")
        # Each item in the list is [20, 6, 1, 1] -> Final is [81, 20, 6, 1, 1]
        tensor_dataset = torch.stack(representative_data, dim=0)
        # Convert to Float32
        float_dataset = tensor_dataset.detach().cpu().numpy().astype(np.float32)
        print(f"Float dataset shape: {float_dataset.shape}")

        m = onnx.load("convlstm.onnx")
        # Print the input shape recorded in the ONNX file
        shape = [d.dim_value for d in m.graph.input[0].type.tensor_type.shape.dim]
        print(f"Onnx input shape: {shape}")

        try:
            # Save the dataset to a numpy file
            np.save(str(npy_file), float_dataset)

        except Exception as e:
            print(f"Error in saving numpy file: {e}")

        if os.path.exists(npy_file):
          print(f"The calibration file is successfully created at {npy_file}. ✅")
        else:
          print(f"The calibration file is not created.")

        np_data = [["video_input", npy_file, [[[[[0, 0, 0, 0, 0, 0]]]]], [[[[[255, 255, 255, 1, 1, 1]]]]]]]
        # np_data = [["video_input", npy_file, [0,0,0,0,0,0], [255,255,255,1,1,1]]]
        return np_data

def get_intent_position():
    # 50% of the dataset have intent
    if random.random() < 0.5:
        # The time positions of the first 1 second (10 fps) because the intent is placed for a second
        start_frame = 0
        end_frame = 9
        median = (start_frame + end_frame)/2
        range_zero = np.arange(-median, median)

        # Obtain the probability of selecting a timestamp using the adjacent 0.5 areas
        smaller_range = range_zero - 0.5
        higher_range = range_zero + 0.5

        # Probability is the difference of the probability of higher range and lower range
        probability = ss.norm.cdf(higher_range) - ss.norm.cdf(smaller_range)

        # Normalize the probabilities
        # Each probability in probability range is divided by the sum of the probabilities in probability range
        probability /= probability.sum()

        # Select a timestamp based on the probabilities
        range = np.arange(start_frame, end_frame)
        intent_position = np.random.choice(range, p=probability)
    else:
        intent_position = -1

    return intent_position

def get_intent(intent_position, df):
        # Check if the data has no intent
        if intent_position != -1:
            intent = get_label(df)
        else:
            intent = -1
        return intent

def get_label(df):
    """Extract label from CSV using majority vote on last 24 frames."""
    col = 'label_id_corrected' if 'label_id_corrected' in df.columns else df.columns[-1]
    counts = df[col].tail(24).value_counts() # CSV files are in 24 fps

    label_counts = [counts.get(i, 0) for i in range(3)]

    if label_counts[0] == 24:
        return 0  # Front
    elif label_counts[1] > label_counts[2]:
        return 1  # Left
    elif label_counts[1] < label_counts[2]:
        return 2  # Right
    else:
        return int(df[col].tail(12).mode().iloc[0])

def get_intent_torch(intent_position, i, intent, frame_tensor, size):
    # Create a tensor for the intent with the same spatial dimensions as the video frames
    # Used for no intent
    intent_torch = torch.zeros((3, size, size))
    # If intent exists, add intent in its intent position for 1 second (10 frames)
    if intent != -1:
        if intent_position != -1 and intent_position <= i and (intent_position + 10) > i:
            # Convert intent to one-hot vector by filling the specified channel with 1
            intent_torch[intent, :, :] = 1

    intent_torch = intent_torch.to(frame_tensor.device)
    # Append the intent as a channel to the video frame
    frame_tensor = torch.cat((frame_tensor, intent_torch), dim=0)

    return frame_tensor

In [5]:
onnx_path = "convlstm.onnx"
tf_dir = "tf_model"

# Convert from ONNX to TensorFlow and TFLite
onnx2tf.convert(
    input_onnx_file_path=onnx_path,
    output_folder_path=tf_dir,
    copy_onnx_input_output_names_to_tflite=True,
    keep_shape_absolutely_input_names=["video_input"], # Keeps the order of input
    keep_ncw_or_nchw_or_ncdhw_input_names=["video_input"],
    # output_dynamic_range_quantized_tflite=True,
    output_integer_quantized_tflite = True,
    # output_integer_quantized_tflite=self.args.int8,
    custom_input_op_name_np_data_path=data_gen(),
    non_verbose=True
)

print(f"Success: conversion to Tensorflow. File is saved at: {tf_dir}")

Dataset unzipped.
Number of files in /content/Dataset_750/videos: 750
Number of items in the representative_data: 100
Float dataset shape: (100, 20, 6, 1, 1)
Onnx input shape: [1, 20, 6, 1, 1]
The calibration file is successfully created at /content/tflite_int8_data.npy. ✅
Saved artifact at 'tf_model'. The following endpoints are available:

* Endpoint 'serving_default'
  inputs_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 20, 6, 1, 1), dtype=tf.float32, name='video_input')
Output Type:
  TensorSpec(shape=(1, 3), dtype=tf.float32, name=None)


Success: conversion to Tensorflow. File is saved at: tf_model


In [10]:
if IN_COLAB:
    # Automated download of TensorFlow and TFLite models
    if os.path.exists(tf_dir):
        !zip -rq {tf_dir}.zip /content/{tf_dir}
        files.download(tf_dir + ".zip")
        print("TensorFlow and TFLite models downloaded ✅")
    else:
        print("TensorFlow and TFLite models were not downloaded.")

    # Automated download of calibration file
    npy_file = "tflite_int8_data.npy"
    if os.path.exists(npy_file):
        files.download(npy_file)
        print(f"Calibration file ({npy_file}) downloaded ✅")
    else:
        print("Calibration file was not downloaded.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

TensorFlow and TFLite models downloaded ✅


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Calibration file (tflite_int8_data.npy) downloaded ✅


In [7]:
# Verifying the TFLite model
tflite_path = os.path.join(tf_dir, "convlstm_float16.tflite")
interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()

output_details = interpreter.get_output_details()[0] # Model has single output.
input_details = interpreter.get_input_details()[0] # Model has single input.
input_shape = input_details['shape']

# Input shape: Batch size, Channels, Height, Width, Timesteps
# Input_data: Batch size, Timesteps, Channels, Height, Width
input_data = get_input(DEVICE, True, None).cpu().numpy().astype(np.float32)
# input_tflite = np.transpose(input_data, (0, 2, 3, 4, 1))
interpreter.set_tensor(input_details['index'], input_data)
interpreter.invoke()

output = interpreter.get_tensor(output_details['index'])
output_shape = output.shape

print("Interpreted TFLite model")
print(f"Input shape: {input_shape}")
print(f"Output shape: {output_shape}")
print(f"Output: {output}")

Interpreted TFLite model
Input shape: [ 1 20  6  1  1]
Output shape: (1, 3)
Output: [[ 0.02035551  0.04228138 -0.06993542]]
